In [1]:
# Welcome to your new notebook
# Type here in the cell editor to add code!
from pyspark.sql import functions as F
spark.conf.set("spark.sql.session.timeZone", "UTC")

KUSTO_URI = "https://trd-s733h2caqmq2f44yrb.z3.kusto.fabric.microsoft.com"
KUSTO_DB = "eh_fleet"
FACT = "gold.fact_alerts"

spark.sql("CREATE SCHEMA IF NOT EXISTS gold")
spark.sql(f"""CREATE TABLE IF NOT EXISTS {FACT} (
  alert_key STRING, alert_category STRING, alert_type STRING, severity STRING, status STRING, is_open BOOLEAN,
  vehicle_key BIGINT, vehicle_id STRING, trip_id STRING, shipment_id STRING, driver_id STRING,
  route_id STRING, cargo_type_id STRING, date_key INT,
  opened_at TIMESTAMP, condition_met_at TIMESTAMP, closed_at TIMESTAMP, duration_minutes DOUBLE,
  observed_value DOUBLE, threshold_value DOUBLE, value_unit STRING, detected_by STRING,
  detected_at_wall TIMESTAMP, trigger_arrival_wall TIMESTAMP, alert_latency_s DOUBLE,
  alert_count INT, message STRING, source_system STRING, loaded_at TIMESTAMP
) USING DELTA""")

# latest state of every alert + the time it was first written
query = """
alert_events
| summarize first_detected_at = min(detected_at), arg_max(last_updated_at, *) by alert_key
| project alert_key, alert_category, alert_type, severity, status, vehicle_id, trip_id, shipment_id, driver_id,
          condition_start_time, condition_met_time, closed_at, observed_value, threshold_value, value_unit,
          detected_by, message, trigger_arrival_time, first_detected_at, last_updated_at
"""
raw = (spark.read.format("com.microsoft.kusto.spark.synapse.datasource")
       .option("accessToken", mssparkutils.credentials.getToken(KUSTO_URI))
       .option("kustoCluster", KUSTO_URI).option("kustoDatabase", KUSTO_DB)
       .option("kustoQuery", query).load())
n_kql = raw.count()
print("alerts in KQL:", n_kql)

if n_kql > 0:
    trips = spark.table("ref.ref_trips").select(F.col("trip_id").alias("t_trip_id"), "route_id", "cargo_type_id")
    df = raw.join(F.broadcast(trips), raw["trip_id"] == F.col("t_trip_id"), "left").drop("t_trip_id")

    # vehicle version valid when the alert started (dim_vehicle comes with Phase 3; until then vehicle_key stays NULL)
    if spark.catalog.tableExists("gold.dim_vehicle"):
        dv = spark.table("gold.dim_vehicle").select(
            "vehicle_key", F.col("vehicle_id").alias("dv_id"),
            F.col("effective_from").alias("dv_from"), F.col("effective_to").alias("dv_to"))
        df = (df.join(dv, (df["vehicle_id"] == F.col("dv_id"))
                          & (df["condition_start_time"] >= F.col("dv_from"))
                          & (df["condition_start_time"] < F.col("dv_to")), "left")
                .drop("dv_id", "dv_from", "dv_to"))
    else:
        df = df.withColumn("vehicle_key", F.lit(None).cast("bigint"))

    out = df.select(
        "alert_key", "alert_category", "alert_type", "severity", "status",
        (F.col("status") == "OPEN").alias("is_open"),
        "vehicle_key", "vehicle_id", "trip_id", "shipment_id", "driver_id", "route_id", "cargo_type_id",
        F.date_format("condition_start_time", "yyyyMMdd").cast("int").alias("date_key"),
        F.col("condition_start_time").alias("opened_at"),
        F.col("condition_met_time").alias("condition_met_at"), "closed_at",
        ((F.unix_timestamp("closed_at") - F.unix_timestamp("condition_start_time")) / 60.0).alias("duration_minutes"),
        F.col("observed_value").cast("double"), F.col("threshold_value").cast("double"),
        "value_unit", "detected_by",
        F.col("first_detected_at").alias("detected_at_wall"),
        F.col("trigger_arrival_time").alias("trigger_arrival_wall"),
        (F.unix_timestamp("first_detected_at") - F.unix_timestamp("trigger_arrival_time")).cast("double").alias("alert_latency_s"),
        F.lit(1).alias("alert_count"), "message", F.lit("KQL").alias("source_system"),
        F.current_timestamp().alias("loaded_at"))
    out.createOrReplaceTempView("alerts_from_kql")

    # KQL rows are overwritten with their newest state; rows written by batch jobs are never touched
    spark.sql(f"""MERGE INTO {FACT} t USING alerts_from_kql s ON t.alert_key = s.alert_key
                  WHEN MATCHED AND t.source_system = 'KQL' THEN UPDATE SET *
                  WHEN NOT MATCHED THEN INSERT *""")

spark.sql(f"SELECT alert_type, status, COUNT(*) AS n FROM {FACT} WHERE source_system = 'KQL' GROUP BY alert_type, status").show()
print("rows in gold from KQL:", spark.sql(f"SELECT COUNT(*) FROM {FACT} WHERE source_system = 'KQL'").first()[0])
mssparkutils.notebook.exit("done")

StatementMeta(, 1860173e-048e-4409-a54d-62b5dc62ec46, 3, Finished, Available, Finished, False)

alerts in KQL: 6
+-----------------+------+---+
|       alert_type|status|  n|
+-----------------+------+---+
|COLD_CHAIN_BREACH|CLOSED|  1|
| EXCESSIVE_IDLING|CLOSED|  2|
|  ROUTE_DEVIATION|CLOSED|  3|
+-----------------+------+---+

rows in gold from KQL: 6


In [ ]:
# Release cached data and clear Spark cache
spark.catalog.clearCache()